Sourced from https://m2lines.github.io/L96_demo/notebooks/intro_ML_and_NNs.html

In [1]:
!nvidia-smi

Fri Jan 30 21:06:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:25:00.0 Off |                    0 |
| N/A   34C    P0             33W /  250W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
from m_star_predictor import (
    LinearRegression, FCNN, fit_model, normalize, real_units,
    reduce_dataset, compute_rmse
)
import torch
import torch.utils.data as Data
from torch.utils.data import random_split
from torch import optim
import numpy as np
import os

import matplotlib.pyplot as plt

# Ensuring reproducibility
np.random.seed(14)
torch.manual_seed(123);
sklear_random_state = 123

In [2]:
if torch.cuda.is_available():
    print("GPU is available.")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. PyTorch is likely using the CPU.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

GPU is available.
Number of GPUs: 1
Current GPU name: NVIDIA A100-PCIE-40GB
Using device: cuda


In [3]:
data_directory = '../m_star_dataset'
dataset_name = 'ePBL_paper_expanded_2283_corrected'
validation_dataset_name = 'ePBL_paper_validation_2283_corrected'
dataset_filename = f"{dataset_name}_dataset_M_target.pt"
validation_dataset_filename = f"{validation_dataset_name}_dataset_M_target.pt"
data = torch.load(os.path.join(data_directory, dataset_filename))
validation_data = torch.load(os.path.join(data_directory, validation_dataset_filename))

In [ ]:
n_layers = 1
n_neurons = 16
weight_decay = 1e-3
learning_rate = 0.003
loss_fn = torch.nn.MSELoss()
n_epochs_fcnn = 120
list_of_indices = [
#     [0,1,2,3,4,5],
#     [0,1,2,3,6],
#     [0,1,2,3,4,5,7],
    [0,1,2,3,4,5,7,8,9,10,11]
#     [0,1,2,3,10,11]
]
list_of_names = [
#     "angle_and_mag",
#     "u_dot_tau",
#     "angle_mag_and_b_hist",
    "bl_included",
#     "ss_nn"
]

for indices, name in zip(list_of_indices, list_of_names):
    X, y = data.T[:,indices].float(), data.T[:,-1].unsqueeze(1).float()
    X_val, y_val = validation_data.T[:,indices].float(), validation_data.T[:,-1].unsqueeze(1).float()

    n_inputs = len(indices)

    TRAIN_BATCH_SIZE = int(1402920 / 16)
    TEST_BATCH_SIZE = 9353
    print(TRAIN_BATCH_SIZE)

    # transform to sqrt
    y_sqrt = y.sqrt()
    y_val_sqrt = y_val.sqrt()

    X_sqrt = X.sqrt()
    X_val_sqrt = X_val.sqrt()

    X_mean = X.mean(dim=0)
    X_std = X.std(dim=0)
    y_mean = y_sqrt.mean(dim=0)
    y_std = 2*y_sqrt.std(dim=0)

    X_train = normalize(data=X, mean=X_mean, std=X_std)
    X_test = normalize(data=X_val, mean=X_mean, std=X_std)
    y_train = normalize(data=y_sqrt, mean=y_mean, std=y_std)
    y_test = normalize(data=y_val_sqrt, mean=y_mean, std=y_std)

    train_dataset = Data.TensorDataset(X_train, y_train)
    test_dataset = Data.TensorDataset(X_test, y_test)

    train_loader = Data.DataLoader(dataset=train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
    test_loader = Data.DataLoader(dataset=test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

    fcnn_network = FCNN(input_size=n_inputs, n_hidden_layers=n_layers, n_neurons=n_neurons)
    optimizer_fcnn = optim.Adam(fcnn_network.parameters(), lr=learning_rate, weight_decay=weight_decay)
    train_loss_fcnn, test_loss_fcnn = fit_model(
        fcnn_network, loss_fn, optimizer_fcnn, train_loader, test_loader, n_epochs_fcnn, device
    )

    fcnn_save_path = f'../m_star_predictors/fcnn_ysqrt_default_{name}_inst_predictor_weights.pth'
    torch.save(fcnn_network.state_dict(), fcnn_save_path)
    fcnn_traj_save_path = f'../m_star_predictors/fcnn_ysqrt_default_{name}_inst_predictor_train_test_loss.npy'
    np.save(fcnn_traj_save_path, np.array([train_loss_fcnn, test_loss_fcnn]))

87682
epoch 1 completed
epoch 2 completed
epoch 3 completed
epoch 4 completed
epoch 5 completed
epoch 6 completed
epoch 7 completed
epoch 8 completed
epoch 9 completed
epoch 10 completed
epoch 11 completed
epoch 12 completed
epoch 13 completed
epoch 14 completed
epoch 15 completed
epoch 16 completed
epoch 17 completed
epoch 18 completed
epoch 19 completed
epoch 20 completed
epoch 21 completed
epoch 22 completed
epoch 23 completed
epoch 24 completed
epoch 25 completed
epoch 26 completed
epoch 27 completed
epoch 28 completed
epoch 29 completed
epoch 30 completed
epoch 31 completed
epoch 32 completed
epoch 33 completed
epoch 34 completed
epoch 35 completed
epoch 36 completed
epoch 37 completed
epoch 38 completed
epoch 39 completed
epoch 40 completed
epoch 41 completed
epoch 42 completed
epoch 43 completed
epoch 44 completed
epoch 45 completed
epoch 46 completed
epoch 47 completed
epoch 48 completed
epoch 49 completed
epoch 50 completed
epoch 51 completed
epoch 52 completed
epoch 53 comple